# Fine-tune YOLOv8n — Two-Earbud Assembly Detector

Notebook dùng để fine-tune YOLOv8 Nano trên dataset đã **gán nhãn lại đúng 5 lớp trạng thái**: `open_case`, `close_case`, `earbud`, `empty_left`, `empty_right`.

> **Quan trọng:** dataset 6 lớp cũ không thể tự suy ra hộp mở/đóng hoặc khe trái/phải. Bạn phải gán lại nhãn trước khi Run All. Notebook sẽ dừng nếu bộ tên lớp chưa đúng để tránh train nhầm.

### Hướng dẫn sử dụng trên Kaggle
1. **Tạo Notebook mới** trên Kaggle.
2. Bật **GPU**: Settings → Accelerator → chọn **GPU T4 x2** hoặc **P100**.
3. **Add Data**: nhấn nút "+Add Data" → Upload → kéo thả file `earbud_merged.zip`.
4. **Import Notebook này**: File → Import Notebook → chọn file `Kaggle_Training_Earbud.ipynb`.
5. Sửa biến `DATASET_ROOT` ở ô đầu tiên cho đúng đường dẫn Kaggle hiển thị sau khi Add Data.
6. **Run All**.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CHỈ CẦN SỬA DATASET_ROOT CHO ĐÚNG ĐƯỜNG DẪN TRÊN KAGGLE     ║
# ╚══════════════════════════════════════════════════════════════════╝

# Đường dẫn đến folder earbud_merged trong Kaggle Input
# (Xem đường dẫn chính xác trong panel bên phải sau khi Add Data)
DATASET_ROOT = '/kaggle/input/datasets/shintheanother/earbud-dtsv1/earbud_merged'

EPOCHS     = 60
BATCH_SIZE = 16   # Giảm xuống 8 nếu GPU bị Out of Memory
IMG_SIZE   = 640
EXPECTED_CLASSES = ['open_case', 'close_case', 'earbud', 'empty_left', 'empty_right']

In [ ]:
!pip install -q ultralytics

In [ ]:
import os, glob, yaml

# Kiểm tra dataset tồn tại
assert os.path.isdir(DATASET_ROOT), (
    f"Không tìm thấy dataset tại: {DATASET_ROOT}\n"
    f"Hãy kiểm tra lại đường dẫn trong panel bên phải."
)
print(f'Dataset root: {DATASET_ROOT}')
print(f'Nội dung: {sorted(os.listdir(DATASET_ROOT))}')

for split in ['train', 'valid', 'test']:
    imgs = glob.glob(os.path.join(DATASET_ROOT, split, 'images', '*'))
    lbls = glob.glob(os.path.join(DATASET_ROOT, split, 'labels', '*'))
    print(f'  {split:5s}: {len(imgs)} anh, {len(lbls)} labels')

# Xác nhận dataset đã được gán lại thật sự. Không được chỉ đổi tên lớp cũ:
# Earphone_Case/Empty_Slot cũ không chứa trạng thái mở-đóng hoặc trái-phải.
source_yaml = os.path.join(DATASET_ROOT, 'data.yaml')
assert os.path.isfile(source_yaml), f'Không tìm thấy data.yaml: {source_yaml}'
with open(source_yaml, encoding='utf-8') as f:
    source_data = yaml.safe_load(f)
source_names = source_data.get('names', [])
if isinstance(source_names, dict):
    source_names = [source_names[i] for i in sorted(source_names)]
assert len(source_names) == 5 and set(source_names) == set(EXPECTED_CLASSES), (
    f'NHÃN DATASET CHƯA ĐÚNG.\nHiện tại: {source_names}\n'
    f'Bắt buộc: {EXPECTED_CLASSES}\n'
    'Hãy gán lại nhãn trên Roboflow/CVAT rồi export YOLOv8.'
)
print('Nhãn dataset hợp lệ (giữ nguyên class ID):', source_names)

# Tạo mới data.yaml với đường dẫn tuyệt đối
yaml_path = '/kaggle/working/data_earbud.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(
        f'path: {DATASET_ROOT}\n'
        'train: train/images\n'
        'val: valid/images\n'
        'test: test/images\n'
        '\n'
        'nc: 5\n'
        f'names: {source_names!r}\n'
    )

print(f'\ndata.yaml da tao tai: {yaml_path}')
print(open(yaml_path).read())

In [ ]:
from ultralytics import YOLO

# File yolov8n.pt đã được đóng gói sẵn trong zip dataset
local_pt = os.path.join(DATASET_ROOT, 'yolov8n.pt')

if os.path.exists(local_pt):
    print(f'Dung pretrained model tu dataset: {local_pt}')
    model = YOLO(local_pt)
else:
    print('Khong tim thay yolov8n.pt trong dataset, dang tai tu Ultralytics...')
    model = YOLO('yolov8n.pt')

print('Model loaded OK')

In [ ]:
# ─── FINE-TUNE ───────────────────────────────────────────────────────────
results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project='/kaggle/working/training',
    name='earbud_merged_detector',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    patience=15,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

In [ ]:
# ─── ĐÁNH GIÁ TRÊN TẬP TEST ─────────────────────────────────────────────
best_model = YOLO('/kaggle/working/training/earbud_merged_detector/weights/best.pt')
metrics = best_model.val(
    data=yaml_path,
    split='test',
    plots=True,
    verbose=True,
)

print('\n════════════════════════════════════════')
print(f'  mAP50     : {metrics.box.map50:.4f}')
print(f'  mAP50-95  : {metrics.box.map:.4f}')
print(f'  Precision : {metrics.box.mp:.4f}')
print(f'  Recall    : {metrics.box.mr:.4f}')
print('════════════════════════════════════════')

In [ ]:
# ─── NÉN KẾT QUẢ ĐỂ TẢI VỀ ──────────────────────────────────────────────
!zip -r -q /kaggle/working/training_results.zip /kaggle/working/training/earbud_merged_detector

import os
size_mb = os.path.getsize('/kaggle/working/training_results.zip') / 1024 / 1024
print(f'\n Da nen ket qua: training_results.zip ({size_mb:.1f} MB)')
print('\nHuong dan tai ve:')
print('  1. Panel ben phai -> muc Output')
print('  2. Tai file training_results.zip')
print('  3. Giai nen, lay file best.pt')
print('  4. Copy de vao: artifacts/training/earbud_merged_detector/weights/best.pt')
print('  5. Chay: python scripts/run_earbud.py  <- camera se dung model moi')